In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

import os
os.makedirs('results', exist_ok=True)

print('✅ All packages imported successfully.')


# ==============================================================================
# TASK 1 — Dataset Understanding
# ==============================================================================

# ── 1.1  Load data ────────────────────────────────────────────────────────────
DATA_PATH = 'customer_support_text_classification.csv'
df = pd.read_csv(DATA_PATH)

print(f'Records  : {len(df):,}')
print(f'Columns  : {df.columns.tolist()}')
print(f'Missing  : {df.isnull().sum().to_dict()}')
print()
df.head(5)

# ── 1.2  Target class distribution ───────────────────────────────────────────
label_counts = df['sentiment_label'].value_counts()
print('Class distribution:')
print(label_counts.to_string())
print(f'\nClass balance (%):')
print((label_counts / len(df) * 100).round(1).to_string())

# ── 1.3  Text length statistics ──────────────────────────────────────────────
df['char_count'] = df['customer_message'].str.len()

print('Word count stats:')
print(df['word_count'].describe().round(2).to_string())
print()
print('Character count stats:')
print(df['char_count'].describe().round(2).to_string())

# ── 1.4  Sample messages per class ───────────────────────────────────────────
for label in ['positive', 'neutral', 'negative']:
    sample = df[df['sentiment_label'] == label]['customer_message'].iloc[0]
    print(f'[{label.upper()}] {sample}')
    print()

# ── 1.5  Visualise class distribution & text length ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
palette = {'positive': '#2ecc71', 'neutral': '#3498db', 'negative': '#e74c3c'}

# Bar chart – class counts
ax = axes[0]
bars = ax.bar(label_counts.index, label_counts.values,
               color=[palette[l] for l in label_counts.index], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, label_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
ax.spines[['top', 'right']].set_visible(False)

# Pie chart
ax = axes[1]
ax.pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
       colors=[palette[l] for l in label_counts.index], startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Class Proportions', fontsize=13, fontweight='bold')

# Box plot – word count per class
ax = axes[2]
for label in ['positive', 'neutral', 'negative']:
    data = df[df['sentiment_label'] == label]['word_count']
    ax.boxplot(data, positions=[['positive','neutral','negative'].index(label)],
               widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=palette[label], alpha=0.7),
               medianprops=dict(color='white', linewidth=2))
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['positive', 'neutral', 'negative'])
ax.set_title('Word Count by Sentiment', fontsize=13, fontweight='bold')
ax.set_ylabel('Word Count')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/eda_plots.png')


# ==============================================================================
# TASK 2 — Text Preprocessing
# ==============================================================================

# ── 2.1  Define a stopword list (built-in, no NLTK dependency) ────────────────
STOPWORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
    "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him',
    'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its',
    'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who',
    'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was',
    'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did',
    'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
    'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into',
    'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down',
    'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once',
    'here', 'there', 'when', 'where', 'why', 'how', 'all', 'both', 'each', 'few', 'more',
    'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so',
    'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should',
    "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't",
    'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't",
    'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't",
    'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't",
    'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"
])

print(f'Stopwords loaded: {len(STOPWORDS)} words')

# ── 2.2  Preprocessing pipeline ──────────────────────────────────────────────
def preprocess_text(text: str, remove_stopwords: bool = True) -> str:
    """
    Full text cleaning pipeline:
      1. Lowercase
      2. Remove URLs, emails, ticket numbers
      3. Remove special characters / punctuation
      4. Collapse whitespace
      5. Tokenise (split on whitespace)
      6. Remove stopwords (optional)
      7. Re-join tokens
    """
    if not isinstance(text, str):
        return ''

    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\b(tkt|ticket|ref|order)?\s?#?\d{4,}\b', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]

    return ' '.join(tokens)


df['clean_message'] = df['customer_message'].apply(preprocess_text)

print('Before:', df['customer_message'].iloc[0])
print('After :', df['clean_message'].iloc[0])
print()
print('Before:', df['customer_message'].iloc[2])
print('After :', df['clean_message'].iloc[2])

# ── 2.3  Tokenisation example ────────────────────────────────────────────────
sample = df['clean_message'].iloc[2]
tokens = sample.split()
print(f'Tokenised ({len(tokens)} tokens): {tokens}')

# ── 2.4  Top tokens per class ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, label in zip(axes, ['positive', 'neutral', 'negative']):
    corpus = ' '.join(df[df['sentiment_label'] == label]['clean_message'])
    freq = Counter(corpus.split()).most_common(15)
    words, counts = zip(*freq)
    color = palette[label]
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'Top 15 Words — {label.capitalize()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Most Frequent Words After Cleaning', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/top_words_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/top_words_per_class.png')


# ==============================================================================
# TASK 3 — Text Vectorisation
# ==============================================================================

# ── 3.1  Prepare labels ───────────────────────────────────────────────────────
le = LabelEncoder()
y = le.fit_transform(df['sentiment_label'])
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X_text = df['clean_message']
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train_text)} | Test: {len(X_test_text)}')

# ── 3.2  Bag of Words ─────────────────────────────────────────────────────────
bow_vectorizer = CountVectorizer(max_features=5000, min_df=2, max_df=0.95)
X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_test_bow  = bow_vectorizer.transform(X_test_text)

print(f'BoW matrix shape (train): {X_train_bow.shape}')
print(f'Sample vocabulary (first 20 words):')
print(bow_vectorizer.get_feature_names_out()[:20].tolist())

# ── 3.3  TF-IDF ───────────────────────────────────────────────────────────────
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)   # unigrams + bigrams
)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_text)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf_vectorizer.vocabulary_):,}')

# ── 3.4  Integer sequence encoding (for sequence models) ─────────────────────
MAX_VOCAB   = 5000
MAX_SEQ_LEN = 50
PAD_ID      = 0
UNK_ID      = 1

all_tokens = ' '.join(X_train_text).split()
vocab_freq  = Counter(all_tokens)
vocab = {word: idx+2 for idx, (word, _) in
         enumerate(vocab_freq.most_common(MAX_VOCAB - 2))}

def text_to_sequence(text, vocab, max_len=MAX_SEQ_LEN):
    tokens = text.split()
    ids = [vocab.get(t, UNK_ID) for t in tokens]
    ids = ids[:max_len]
    ids = ids + [PAD_ID] * (max_len - len(ids))
    return ids

X_train_seq = np.array([text_to_sequence(t, vocab) for t in X_train_text])
X_test_seq  = np.array([text_to_sequence(t, vocab) for t in X_test_text])

print(f'Sequence matrix shape: {X_train_seq.shape}')
print(f'Sample sequence (first message): {X_train_seq[0]}')


# ==============================================================================
# TASK 4 — Baseline Models
# ==============================================================================

# ── 4.1  Naive Bayes with Bag of Words ────────────────────────────────────────
nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_train_bow, y_train)
y_pred_nb = nb_model.predict(X_test_bow)

print('=' * 55)
print('Naive Bayes (Bag of Words)')
print('=' * 55)
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))

# ── 4.2  Logistic Regression with TF-IDF ─────────────────────────────────────
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

print('=' * 55)
print('Logistic Regression (TF-IDF + Bigrams)')
print('=' * 55)
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

# ── 4.3  LinearSVC with TF-IDF ────────────────────────────────────────────────
svc_model = LinearSVC(max_iter=2000, C=0.5, random_state=42)
svc_model.fit(X_train_tfidf, y_train)
y_pred_svc = svc_model.predict(X_test_tfidf)

print('=' * 55)
print('LinearSVC (TF-IDF + Bigrams)')
print('=' * 55)
print(classification_report(y_test, y_pred_svc, target_names=le.classes_))

# ── 4.4  Compare models visually ─────────────────────────────────────────────
models = {
    'Naive Bayes\n(BoW)': y_pred_nb,
    'Logistic Reg\n(TF-IDF)': y_pred_lr,
    'LinearSVC\n(TF-IDF)': y_pred_svc,
}

metrics = {}
for name, y_pred in models.items():
    metrics[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 (macro)': f1_score(y_test, y_pred, average='macro'),
        'F1 (weighted)': f1_score(y_test, y_pred, average='weighted'),
    }

metrics_df = pd.DataFrame(metrics).T
print(metrics_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(metrics_df))
w = 0.25
for i, col in enumerate(metrics_df.columns):
    ax.bar(x + i*w, metrics_df[col], width=w, label=col, alpha=0.85)
ax.set_xticks(x + w)
ax.set_xticklabels(metrics_df.index)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Baseline Model Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/model_comparison.png')

# ── 4.5  Best model confusion matrix ─────────────────────────────────────────
best_name = metrics_df['F1 (macro)'].idxmax()
best_pred = models[best_name]
print(f'Best model: {best_name.replace(chr(10), " ")}\n')

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name.replace(chr(10), " ")}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/confusion_matrix.png')

# ── 4.6  Save evaluation results to CSV ──────────────────────────────────────
metrics_df.to_csv('results/model_evaluation.csv')
print('Saved → results/model_evaluation.csv')

# ── 4.7  Sample predictions ──────────────────────────────────────────────────
sample_messages = [
    "The service was absolutely fantastic, I'm so happy!",
    "My order hasn't arrived yet. This is completely unacceptable.",
    "I need to know the status of my ticket.",
    "Great support team, resolved my issue in minutes!",
    "Still waiting for a refund after two weeks. Very disappointed.",
]

cleaned_samples = [preprocess_text(m) for m in sample_messages]
vec_samples     = tfidf_vectorizer.transform(cleaned_samples)
preds           = lr_model.predict(vec_samples)
pred_labels     = le.inverse_transform(preds)

print('Sample Predictions (Logistic Regression / TF-IDF):')
print('-' * 60)
lines = []
for msg, pred in zip(sample_messages, pred_labels):
    line = f'[{pred.upper():8s}]  {msg}'
    print(line)
    lines.append(line)

with open('results/sample_predictions.txt', 'w') as f:
    f.write('Sample Predictions (Logistic Regression / TF-IDF)\n')
    f.write('-' * 60 + '\n')
    f.write('\n'.join(lines))
print('\nSaved → results/sample_predictions.txt')


# ==============================================================================
# TASK 5 — Sequence Model Architecture (LSTM)
# ==============================================================================

# ── 5.1  Architecture summary ─────────────────────────────────────────────────
architecture = {
    'Layer 1': 'Embedding     | input_dim=5002, output_dim=64, input_length=50',
    'Layer 2': 'SpatialDropout| rate=0.2  (regularisation on embedding)',
    'Layer 3': 'LSTM          | units=64, return_sequences=False, dropout=0.2',
    'Layer 4': 'Dense         | units=32, activation=relu',
    'Layer 5': 'Dropout       | rate=0.3',
    'Layer 6': 'Dense (output)| units=3,  activation=softmax',
}

print('LSTM Architecture for Sentiment Classification')
print('=' * 60)
for layer, desc in architecture.items():
    print(f'  {layer}: {desc}')
print('=' * 60)
print()
print('Compilation settings:')
print('  Loss function : sparse_categorical_crossentropy')
print('  Optimiser     : Adam (lr=0.001)')
print('  Metrics       : accuracy, F1 (macro)')
print('  Epochs        : 10  |  Batch size : 32')
print('  Early stopping: patience=3, monitor=val_loss')

# ── 5.2  Keras model code (reference — requires TensorFlow) ───────────────────
lstm_code = '''
# Install: pip install tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping

VOCAB_SIZE  = 5002
EMBED_DIM   = 64
SEQ_LEN     = 50
NUM_CLASSES = 3

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
              input_length=SEQ_LEN, mask_zero=True),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.1),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train_seq, y_train,
    validation_split=0.15,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

loss, acc = model.evaluate(X_test_seq, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")
'''
print(lstm_code)

# ── 5.3  Forward pass simulation (NumPy — no TF needed) ───────────────────────
np.random.seed(42)

VOCAB_SIZE  = 5002
EMBED_DIM   = 64
SEQ_LEN     = 50
NUM_CLASSES = 3

embedding_matrix = np.random.randn(VOCAB_SIZE, EMBED_DIM) * 0.01

def embed(seq):
    return embedding_matrix[seq]

def lstm_step(x_t, h_prev, c_prev, W, U, b):
    z = x_t @ W + h_prev @ U + b
    units = W.shape[1] // 4
    i = 1 / (1 + np.exp(-z[:units]))
    f = 1 / (1 + np.exp(-z[units:2*units]))
    g = np.tanh(z[2*units:3*units])
    o = 1 / (1 + np.exp(-z[3*units:]))
    c = f * c_prev + i * g
    h = o * np.tanh(c)
    return h, c

LSTM_UNITS = 64
W  = np.random.randn(EMBED_DIM,  4 * LSTM_UNITS) * 0.01
U  = np.random.randn(LSTM_UNITS, 4 * LSTM_UNITS) * 0.01
b  = np.zeros(4 * LSTM_UNITS)
W2 = np.random.randn(LSTM_UNITS, NUM_CLASSES) * 0.01
b2 = np.zeros(NUM_CLASSES)

sample_seq = X_train_seq[0]
embedded   = embed(sample_seq)

h = np.zeros(LSTM_UNITS)
c = np.zeros(LSTM_UNITS)
for t in range(SEQ_LEN):
    h, c = lstm_step(embedded[t], h, c, W, U, b)

logits     = h @ W2 + b2
exp_logits = np.exp(logits - logits.max())
probs      = exp_logits / exp_logits.sum()

print('LSTM Forward Pass Simulation (random weights)')
print(f'  Input sequence length : {SEQ_LEN}')
print(f'  Embedding dim         : {EMBED_DIM}')
print(f'  LSTM hidden units     : {LSTM_UNITS}')
print(f'  Output (softmax probs): {dict(zip(le.classes_, probs.round(4)))}')
print(f'  Predicted class       : {le.classes_[probs.argmax()]} (random — untrained weights)')

# ── 5.4  Architecture diagram (matplotlib) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

layers = [
    ('Input Sequences\n(50 integer IDs)', '#95a5a6'),
    ('Embedding Layer\n(5002 → 64 dims)', '#3498db'),
    ('SpatialDropout\n(rate=0.2)', '#85c1e9'),
    ('LSTM Layer\n(64 units)', '#9b59b6'),
    ('Dense Layer\n(32 units, ReLU)', '#e67e22'),
    ('Dropout\n(rate=0.3)', '#f0b27a'),
    ('Output Layer\n(3 units, Softmax)', '#2ecc71'),
]

y_positions = np.linspace(6.8, 0.8, len(layers))
box_w, box_h = 3.5, 0.65
cx = 5

for (label, color), yp in zip(layers, y_positions):
    rect = mpatches.FancyBboxPatch(
        (cx - box_w/2, yp - box_h/2), box_w, box_h,
        boxstyle='round,pad=0.05', linewidth=1.5,
        edgecolor='white', facecolor=color, alpha=0.88,
    )
    ax.add_patch(rect)
    ax.text(cx, yp, label, ha='center', va='center',
            fontsize=9.5, fontweight='bold', color='white')

for i in range(len(layers) - 1):
    y_start = y_positions[i]   - box_h/2
    y_end   = y_positions[i+1] + box_h/2
    ax.annotate('', xy=(cx, y_end), xytext=(cx, y_start),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

ax.set_title('LSTM Architecture for Sentiment Classification',
             fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('results/lstm_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/lstm_architecture.png')


# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
print('=' * 60)
print('PIPELINE SUMMARY')
print('=' * 60)
print(f'  Dataset        : {len(df):,} customer support messages')
print(f'  Classes        : {list(le.classes_)}')
print(f'  Avg word count : {df["word_count"].mean():.1f}')
print()
print('  Vectorisation methods: BoW, TF-IDF (bigrams), Integer Sequences')
print()
best_acc = metrics_df['Accuracy'].max()
best_f1  = metrics_df['F1 (macro)'].max()
print(f'  Best baseline accuracy : {best_acc:.4f}')
print(f'  Best baseline F1 (mac) : {best_f1:.4f}')
print()
print('  Sequence model : LSTM (architecture defined; NumPy simulation run)')
print('  Results saved  → results/')
print('=' * 60)